# Poverty and Child Poverty Rates by State

This notebook calculates SPM poverty and child poverty rates for all 50 US states plus Washington DC using PolicyEngine microsimulation.

**Methodology:**
- Uses `person_in_poverty` variable (SPM poverty definition)
- Child poverty: persons under age 18 in poverty
- Weighted calculations using state-specific datasets

In [1]:
from policyengine_us import Microsimulation
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
import plotly.express as px
import plotly.graph_objects as go
import gc  # For memory management

In [2]:
# Configuration
ANALYSIS_YEAR = 2026

# States ordered by approximate population (smallest first) to reduce early memory pressure
# Large states (CA, TX, FL, NY) are processed last
STATES = [
    "WY", "VT", "DC", "AK", "ND", "SD", "DE", "RI", "MT", "ME",
    "NH", "HI", "WV", "ID", "NE", "NM", "KS", "MS", "AR", "NV",
    "IA", "UT", "CT", "OK", "OR", "KY", "LA", "AL", "SC", "MN",
    "CO", "WI", "MD", "MO", "TN", "AZ", "IN", "MA", "WA", "VA",
    "NJ", "MI", "NC", "GA", "OH", "IL", "PA", "NY", "FL", "TX",
    "CA"
]

# State names for display
STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming", "DC": "District of Columbia"
}

In [3]:
def get_state_dataset(state: str) -> str:
    """Download state-specific dataset from Hugging Face."""
    filename = f"states/{state}.h5"
    dataset_path = hf_hub_download(
        repo_id="policyengine/policyengine-us-data",
        filename=filename,
        repo_type="model",
    )
    return dataset_path


def calculate_poverty_rates(state: str, year: int = ANALYSIS_YEAR) -> dict:
    """
    Calculate poverty and child poverty rates for a state.
    
    Uses person_in_poverty variable (SPM poverty definition).
    Child poverty: age < 18.
    
    Memory-optimized: extracts arrays immediately and cleans up simulation.
    """
    # Get state dataset and run simulation
    dataset_path = get_state_dataset(state)
    sim = Microsimulation(dataset=dataset_path)
    
    # Extract arrays immediately to minimize memory footprint
    poverty_arr = np.array(sim.calculate("person_in_poverty", year).values, dtype=np.float32)
    age_arr = np.array(sim.calculate("age", year).values, dtype=np.float32)
    weight_arr = np.array(sim.calculate("person_weight", year).values, dtype=np.float32)
    
    # Delete simulation object and force garbage collection
    del sim
    gc.collect()
    
    # Overall poverty rate (weighted mean)
    poverty_rate = float(np.average(poverty_arr, weights=weight_arr))
    
    # Child poverty rate (age < 18)
    child_mask = age_arr < 18
    
    if weight_arr[child_mask].sum() > 0:
        child_poverty_rate = float(np.average(poverty_arr[child_mask], weights=weight_arr[child_mask]))
    else:
        child_poverty_rate = 0.0
    
    # Population counts
    total_population = float(weight_arr.sum())
    child_population = float(weight_arr[child_mask].sum())
    population_in_poverty = float((poverty_arr * weight_arr).sum())
    children_in_poverty = float((poverty_arr[child_mask] * weight_arr[child_mask]).sum())
    
    # Clean up arrays
    del poverty_arr, age_arr, weight_arr, child_mask
    gc.collect()
    
    return {
        "state": state,
        "state_name": STATE_NAMES[state],
        "poverty_rate": poverty_rate,
        "child_poverty_rate": child_poverty_rate,
        "total_population": total_population,
        "child_population": child_population,
        "population_in_poverty": population_in_poverty,
        "children_in_poverty": children_in_poverty,
    }

In [4]:
# Calculate poverty rates for all states with memory optimization
# Set RESUME=True to continue from checkpoint if kernel crashed
RESUME = False

intermediate_file = f"state_poverty_rates_{ANALYSIS_YEAR}_partial.csv"

# Load checkpoint if resuming
results = []
completed_states = set()
if RESUME:
    try:
        checkpoint_df = pd.read_csv(intermediate_file)
        results = checkpoint_df.to_dict('records')
        completed_states = set(checkpoint_df['state'].tolist())
        print(f"Resumed from checkpoint: {len(results)} states already completed")
    except FileNotFoundError:
        print("No checkpoint found, starting fresh")

print(f"Calculating poverty rates for {len(STATES)} jurisdictions...\n")

for i, state in enumerate(STATES):
    if state in completed_states:
        print(f"[{i+1}/{len(STATES)}] {STATE_NAMES[state]} - already completed, skipping")
        continue
        
    print(f"[{i+1}/{len(STATES)}] Processing {STATE_NAMES[state]}...", end=" ")
    try:
        state_results = calculate_poverty_rates(state, ANALYSIS_YEAR)
        results.append(state_results)
        print(f"Poverty: {state_results['poverty_rate']:.1%}, Child: {state_results['child_poverty_rate']:.1%}")
        
        # Save checkpoint every 5 new states
        if len(results) % 5 == 0:
            pd.DataFrame(results).to_csv(intermediate_file, index=False)
            print(f"    [Checkpoint saved: {len(results)} states]")
            
    except Exception as e:
        print(f"ERROR: {e}")
    
    # Force garbage collection after each state
    gc.collect()

# Final save
if results:
    pd.DataFrame(results).to_csv(intermediate_file, index=False)
    
print(f"\nCompleted {len(results)} of {len(STATES)} states.")

Calculating poverty rates for 51 jurisdictions...

[1/51] Processing Wyoming... Poverty: 17.8%, Child: 16.0%


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


[2/51] Processing Vermont... 

VT.h5:   0%|          | 0.00/8.84M [00:00<?, ?B/s]

Poverty: 24.8%, Child: 21.3%
[3/51] Processing District of Columbia... Poverty: 36.8%, Child: 29.2%
[4/51] Processing Alaska... Poverty: 17.1%, Child: 12.5%


...

Poverty: 30.4%, Child: 29.2%
    [Checkpoint saved: 50 states]


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


[51/51] Processing California... 

CA.h5:   0%|          | 0.00/473M [00:00<?, ?B/s]

ERROR: Unable to allocate 1.21 MiB for an array with shape (316717,) and data type float32

Completed 50 of 51 states.


In [5]:
# Create DataFrame with results
df = pd.DataFrame(results)

if len(df) == 0:
    print("No results to display. Check for errors above.")
else:
    # Format for display
    df["poverty_rate_pct"] = df["poverty_rate"] * 100
    df["child_poverty_rate_pct"] = df["child_poverty_rate"] * 100

    # Sort by poverty rate (highest first)
    df_sorted = df.sort_values("poverty_rate", ascending=False).reset_index(drop=True)

    print(f"\n{'='*80}")
    print(f"POVERTY RATES BY STATE ({ANALYSIS_YEAR}) - {len(df)} states completed")
    print(f"{'='*80}")
    print(f"{'Rank':<6} {'State':<25} {'Poverty Rate':<15} {'Child Poverty Rate':<20}")
    print("-" * 80)
    for i, row in df_sorted.iterrows():
        print(f"{i+1:<6} {row['state_name']:<25} {row['poverty_rate_pct']:>10.1f}% {row['child_poverty_rate_pct']:>15.1f}%")


POVERTY RATES BY STATE (2026) - 50 states completed
Rank   State                     Poverty Rate    Child Poverty Rate  
--------------------------------------------------------------------------------
1      District of Columbia            36.8%            29.2%
2      New York                        35.4%            28.8%
3      Oregon                          34.3%            23.1%
4      Hawaii                          33.4%            25.9%
5      Massachusetts                   33.1%            23.4%
6      Georgia                         31.9%            27.9%
7      Florida                         31.8%            34.9%
8      Maryland                        31.1%            25.8%
9      Texas                           30.4%            29.2%
10     Mississippi                     29.3%            25.5%
11     Delaware                        29.3%            25.1%
12     Louisiana                       29.3%            26.2%
13     Virginia                        28.9%        

In [6]:
# Summary statistics
if len(df) > 0:
    print(f"\n{'='*60}")
    print(f"SUMMARY STATISTICS ({len(df)} states)")
    print(f"{'='*60}")

    print(f"\nOverall Poverty Rate:")
    print(f"  Mean:   {df['poverty_rate_pct'].mean():.1f}%")
    print(f"  Median: {df['poverty_rate_pct'].median():.1f}%")
    print(f"  Min:    {df['poverty_rate_pct'].min():.1f}% ({df.loc[df['poverty_rate_pct'].idxmin(), 'state_name']})")
    print(f"  Max:    {df['poverty_rate_pct'].max():.1f}% ({df.loc[df['poverty_rate_pct'].idxmax(), 'state_name']})")

    print(f"\nChild Poverty Rate:")
    print(f"  Mean:   {df['child_poverty_rate_pct'].mean():.1f}%")
    print(f"  Median: {df['child_poverty_rate_pct'].median():.1f}%")
    print(f"  Min:    {df['child_poverty_rate_pct'].min():.1f}% ({df.loc[df['child_poverty_rate_pct'].idxmin(), 'state_name']})")
    print(f"  Max:    {df['child_poverty_rate_pct'].max():.1f}% ({df.loc[df['child_poverty_rate_pct'].idxmax(), 'state_name']})")

    # Population-weighted national average (based on completed states)
    national_poverty = (df['population_in_poverty'].sum() / df['total_population'].sum()) * 100
    national_child_poverty = (df['children_in_poverty'].sum() / df['child_population'].sum()) * 100

    print(f"\nPopulation-Weighted Average (across {len(df)} states):")
    print(f"  Poverty rate:       {national_poverty:.1f}%")
    print(f"  Child poverty rate: {national_child_poverty:.1f}%")


SUMMARY STATISTICS (50 states)

Overall Poverty Rate:
  Mean:   26.0%
  Median: 26.5%
  Min:    12.5% (New Hampshire)
  Max:    36.8% (District of Columbia)

Child Poverty Rate:
  Mean:   21.5%
  Median: 21.6%
  Min:    12.5% (Alaska)
  Max:    34.9% (Florida)

Population-Weighted Average (across 50 states):
  Poverty rate:       28.0%
  Child poverty rate: 23.9%


In [7]:
# Visualization: Bar chart of poverty rates
df_plot = df_sorted.head(20)  # Top 20 highest poverty

fig = go.Figure()

fig.add_trace(go.Bar(
    name="Overall Poverty",
    x=df_plot["state"],
    y=df_plot["poverty_rate_pct"],
    marker_color="#105293",
))

fig.add_trace(go.Bar(
    name="Child Poverty",
    x=df_plot["state"],
    y=df_plot["child_poverty_rate_pct"],
    marker_color="#F2994A",
))

fig.update_layout(
    title=f"Top 20 States by Poverty Rate ({ANALYSIS_YEAR})",
    xaxis_title="State",
    yaxis_title="Poverty Rate (%)",
    barmode="group",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
)

fig.show()

In [8]:
# Visualization: Choropleth map of poverty rates
fig_map = px.choropleth(
    df,
    locations="state",
    locationmode="USA-states",
    color="poverty_rate_pct",
    scope="usa",
    color_continuous_scale="Reds",
    labels={"poverty_rate_pct": "Poverty Rate (%)"},
    title=f"SPM Poverty Rate by State ({ANALYSIS_YEAR})",
    hover_name="state_name",
    hover_data={"poverty_rate_pct": ":.1f", "child_poverty_rate_pct": ":.1f", "state": False},
)

fig_map.update_layout(
    geo=dict(bgcolor="rgba(0,0,0,0)"),
)

fig_map.show()

In [9]:
# Visualization: Child poverty choropleth
fig_child_map = px.choropleth(
    df,
    locations="state",
    locationmode="USA-states",
    color="child_poverty_rate_pct",
    scope="usa",
    color_continuous_scale="Oranges",
    labels={"child_poverty_rate_pct": "Child Poverty Rate (%)"},
    title=f"SPM Child Poverty Rate by State ({ANALYSIS_YEAR})",
    hover_name="state_name",
    hover_data={"poverty_rate_pct": ":.1f", "child_poverty_rate_pct": ":.1f", "state": False},
)

fig_child_map.update_layout(
    geo=dict(bgcolor="rgba(0,0,0,0)"),
)

fig_child_map.show()

In [10]:
# Export to CSV
import os

output_df = df[["state", "state_name", "poverty_rate_pct", "child_poverty_rate_pct", 
                "total_population", "child_population", "population_in_poverty", "children_in_poverty"]].copy()
output_df.columns = ["State Code", "State Name", "Poverty Rate (%)", "Child Poverty Rate (%)",
                     "Total Population", "Child Population", "Population in Poverty", "Children in Poverty"]
output_df = output_df.sort_values("Poverty Rate (%)", ascending=False)

# Save final CSV
output_filename = f"state_poverty_rates_{ANALYSIS_YEAR}.csv"
output_df.to_csv(output_filename, index=False)
print(f"Results saved to {output_filename}")

# Clean up intermediate checkpoint file
if os.path.exists(intermediate_file):
    os.remove(intermediate_file)
    print(f"Removed checkpoint file: {intermediate_file}")

Results saved to state_poverty_rates_2026.csv
Removed checkpoint file: state_poverty_rates_2026_partial.csv


In [11]:
# Display full table
display_df = output_df.copy()
display_df["Poverty Rate (%)"] = display_df["Poverty Rate (%)"].round(1)
display_df["Child Poverty Rate (%)"] = display_df["Child Poverty Rate (%)"].round(1)
display_df["Total Population"] = display_df["Total Population"].apply(lambda x: f"{x:,.0f}")
display_df["Child Population"] = display_df["Child Population"].apply(lambda x: f"{x:,.0f}")
display_df["Population in Poverty"] = display_df["Population in Poverty"].apply(lambda x: f"{x:,.0f}")
display_df["Children in Poverty"] = display_df["Children in Poverty"].apply(lambda x: f"{x:,.0f}")
display_df

,State Code,State Name,Poverty Rate (%),Child Poverty Rate (%),Total Population,Child Population,Population in Poverty,Children in Poverty
2,DC,District of Columbia,36.8,29.2,"688,772","144,009","253,442","42,078"
47,NY,New York,35.4,28.8,"19,979,982","4,335,587","7,068,904","1,248,466"
24,OR,Oregon,34.3,23.1,"4,285,052","897,387","1,467,666","207,090"
11,HI,Hawaii,33.4,25.9,"1,451,218","315,645","484,733","81,702"
37,MA,Massachusetts,33.1,23.4,"7,142,086","1,502,289","2,363,096","350,937"
43,GA,Georgia,31.9,27.9,"11,219,642","2,723,662","3,583,364","760,523"
48,FL,Florida,31.8,34.9,"23,400,412","4,808,358","7,451,507","1,678,164"
32,MD,Maryland,31.1,25.8,"6,271,177","1,467,694","1,953,293","379,143"
49,TX,Texas,30.4,29.2,"31,307,682","8,122,860","9,528,714","2,370,564"
17,MS,Mississippi,29.3,25.5,"2,976,668","726,285","872,930","185,345"
